In [17]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [21]:
from src import fetch
import pandas as pd
import json

data=fetch.get_json('https://www.rebas.tw/api/formal/players/PtGe1/seasons/CPBL-2026-oB/logs')['data']
print(f'{len(data)}場')
game=data[0]
print(game.keys())
pa=game['PA_list'][0]
print(pa)
print({k:v for k, v in pa.items() if k not in ('events','ball_counts')})

25場
dict_keys(['PA_list', 'away', 'date', 'home', 'location_abbr', 'opponent_abbr', 'opponent_uniqid', 'pitching', 'seq', 'strikezone_completeness', 'uniqid'])
{'LI': '', 'PA_order': 6, 'PA_round': 3, 'RBI': 0, 'RE': '0.504', 'RE24': '-0.239', 'SP_game_score': '', 'WPA': '-0.035', 'away_score': 0, 'b_hand': 'L', 'ball_counts': ['S', 'H'], 'balls': 0, 'bases': 0, 'batter': {'name': '高捷', 'number': '4', 'uniqid': 'SCI7M'}, 'endOuts': 1, 'end_away_score': 0, 'end_home_score': 0, 'events': [{'in_play': False, 'is_ball': False, 'is_scoring': False, 'is_strike': False, 'pitch': None, 'type': 'SUB'}, {'in_play': False, 'is_ball': False, 'is_scoring': False, 'is_strike': True, 'pitch': {'EV': '', 'RPM': '2449', 'code': 'S', 'coord_x': '57.78', 'coord_y': '22.08', 'fielder_position': '', 'hardness': '', 'location_code': '', 'location_coord_x': '', 'location_coord_y': '', 'trajectory': '', 'type': 'FC', 'velocity': '132'}, 'type': 'PITCH'}, {'in_play': True, 'is_ball': False, 'is_scoring': False

In [20]:
%%writefile -a src/fetch.py

# ---------- Day 8：逐打席紀錄 ----------
pa_path=Path(__file__).resolve().parent.parent/"data"/"raw"/"pa"

def flatten_pa(logs):
    rows=[]
    for game in logs:
        for pa in game.get('PA_list',[]):
            pitches=[e['pitch'] for e in pa.get('events',[]) if e.get('type')=='PITCH' and e.get('pitch')]
            rows.append({
                'game_id':game['uniqid'],
                'date':game['date'],
                'inning':pa.get('inning'),
                'pa_round':pa.get('PA_round'),
                'batter_id':pa['batter']['uniqid'],
                'batter':pa['batter']['name'],
                'pitcher_id':pa['pitcher']['uniqid'],
                'pitcher':pa['pitcher']['name'],
                'b_hand':pa.get('b_hand'),
                'p_hand':pa.get('p_hand'),
                'result':pa.get('result'),
                'pitch_types':','.join(p.get('type') or '' for p in pitches),
                'pitch_codes':','.join(p.get('code') or '' for p in pitches)
            })
    return rows


Appending to src/fetch.py


In [22]:
%%writefile -a src/fetch.py

def fetch_pitcher_pa(player_id,season=SEASON,refresh=False):
    pa_path.mkdir(parents=True,exist_ok=True)
    file=pa_path/f'{season}_{player_id}.csv'
    if file.exists() and not refresh:
        df=pd.read_csv(file,dtype=str,keep_default_na=False)
    else:
        data_list=get_json(f'https://www.rebas.tw/api/formal/players/{player_id}/seasons/{season}/logs')['data']
        df=pd.DataFrame(flatten_pa(data_list)).astype(str)
        df.to_csv(file,index=False)
    # 野手兼投手（例如林子偉）的紀錄裡也有他當打者的打席，只留他當投手的
    return df[df['pitcher_id']==player_id]


Appending to src/fetch.py


In [23]:
%%writefile scripts/fetch_all_pa.py
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import pandas as pd
from src import fetch

pitchers=fetch.fetch_player('pitcher',refresh=False)
frames=[]
for i , pid in enumerate(pitchers['player_id'],1):
    frames.append(fetch.fetch_pitcher_pa(pid))
    if i % 20 == 0:
        print(f'{i}/{len(pitchers)}')

df_all_pitcher=pd.concat(frames,ignore_index=True)
out=Path(__file__).resolve().parent.parent/ "data" / "clean" / "pa.csv"
out.parent.mkdir(parents=True, exist_ok=True)
df_all_pitcher.to_csv(out,index=False)
print(f'完成：{len(df_all_pitcher)} 個打席 → {out}')


Writing scripts/fetch_all_pa.py


In [5]:
import pandas as pd
csv=pd.read_csv('data/clean/pa.csv',dtype=str,keep_default_na=False)

print(len(csv))
print(csv['result'].value_counts().head(5))
print(csv["b_hand"].value_counts())

27268
result
FO     7165
GO     4889
1B     4768
SO     4319
uBB    2108
Name: count, dtype: int64
b_hand
L    15535
R    11733
Name: count, dtype: int64


In [1]:
!git add .
!git commit -m "Day 8: 逐打席資料"

[main b3858ac] Day 8: 逐打席資料
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 21 files changed, 28658 insertions(+), 745 deletions(-)
 create mode 100644 .virtual_documents/D05.ipynb
 create mode 100644 .virtual_documents/D06.ipynb
 create mode 100644 data/clean/batters.csv
 create mode 100644 data/clean/half_standings.csv
 create mode 100644 data/clean/league_constants.csv
 create mode 100644 data/clean/pa.csv
 create mode 100644 data/clean/pitchers.csv
 create mode 100644 data/clean/teams.csv
 create mode 100644 scripts/fetch_all_pa.py


In [12]:
%%writefile src/splits.py
import pandas as pd
from src.metrics import W

# 打席結果代碼 → wOBA 權重（沒列出的都是出局，權重 0）
woba_weight={"uBB": W["ubb"], "HBP": W["wHBP"], "1B": W["w1B"],
             "2B": W["w2B"], "3B": W["w3B"], "HR": W["wHR"]}
# 不算進 wOBA 分母的打席：故意四壞、犧牲觸擊、資料空白
not_count={'SH','SH_E','IBB','','nan'}

def prepare(pa,scale):
    pa=pa.copy()
    pa['result']=pa['result'].fillna('').astype(str)
    pa=pa[~pa['result'].isin(not_count)]
    pa['woba_val']=pa['result'].map(woba_weight).fillna(0)*scale
    pa['pa_round']=pd.to_numeric(pa['pa_round'], errors='coerce').clip(upper=3)
    return pa

def summarize(pa,keys):
    g=pa.groupby(keys)
    return g.agg(PA=('woba_val','size'),wOBA=('woba_val','mean')).reset_index()


Overwriting src/splits.py


In [13]:
from src import splits

import pandas as pd
pa=pd.read_csv('data/clean/pa.csv',dtype=str,keep_default_na=False)
value=pd.read_csv('data/clean/league_constants.csv',index_col=0).iloc[:,0]
print(pa)
print(value)
p=splits.prepare(pa,value['woba_scale'])
s=splits.summarize(p,["batter_id", "p_hand"])


      game_id        date inning pa_round batter_id batter pitcher_id pitcher  \
0       6j9ao  2026-10-01      1        1     Nf4Ac    邱智呈      QyYQE      蔣銲   
1       6j9ao  2026-10-01      1        1     54TAw    張皓崴      QyYQE      蔣銲   
2       6j9ao  2026-10-01      1        1     LsMyw    潘傑楷      QyYQE      蔣銲   
3       6j9ao  2026-10-01      1        1     Y0UJM    林子豪      QyYQE      蔣銲   
4       6j9ao  2026-10-01      2        1     BMBs8    林培緯      QyYQE      蔣銲   
...       ...         ...    ...      ...       ...    ...        ...     ...   
27263   pTstr  2026-09-13      8        4     LlGVK    陳傑憲      Z4dyf     王定穎   
27264   pTstr  2026-09-13      8        4     Nfm2a    李丞齡      Z4dyf     王定穎   
27265   pTstr  2026-09-13      8        4     LsMyw    潘傑楷      Z4dyf     王定穎   
27266   pTstr  2026-09-13      8        4     BMBs8    林培緯      Z4dyf     王定穎   
27267   pTstr  2026-09-13      8        4     zbFte    林祖傑      Z4dyf     王定穎   

      b_hand p_hand result 

In [14]:
wide=s.pivot(index='batter_id',columns='p_hand',values=['PA','wOBA'])
wide.columns=[f'{a}_vs{b}' for a, b in wide.columns]
print(wide)

           PA_vsL  PA_vsR  wOBA_vsL  wOBA_vsR
batter_id                                    
00Or0        25.0    40.0  0.221252  0.359144
0UuMJ         1.0     1.0  0.000000  0.950911
0YgQ0        70.0   273.0  0.316446  0.313612
0fd4d        64.0   202.0  0.429568  0.291849
0uRxO        61.0   181.0  0.466447  0.275291
...           ...     ...       ...       ...
zGfee        25.0    30.0  0.190182  0.184377
zRABu         1.0     8.0  0.000000  0.118864
zaYo4        29.0    79.0  0.305983  0.264937
zbFte        51.0    38.0  0.288551  0.396538
zpOzg        22.0    47.0  0.259339  0.212331

[183 rows x 4 columns]


In [39]:

a=splits.summarize(p,['pitcher_id','b_hand'])
a_pivot=a.pivot(index='pitcher_id',columns='b_hand',values=['PA','wOBA'])
a_pivot.columns=[f'{a}_vs{b}' for a,b in a_pivot.columns]
k=splits.summarize(p,['pitcher_id','pa_round'])
k_pivot=k.pivot(index='pitcher_id',columns='pa_round',values='wOBA')
k_pivot.columns=[f'woba_round{int(c)}' for c in k_pivot.columns]

pitchers=pd.read_csv('data/clean/pitchers.csv')

pitchers=pitchers[(pitchers['role']=='先發')&(pitchers['IP']>=60)]
pitcher_merge=pitchers.merge(k_pivot,left_on='player_id',right_index=True)
pitcher_merge['gap']=pitcher_merge['woba_round3']-pitcher_merge['woba_round1']
print(pitcher_merge.columns.to_list)
print(pitcher_merge.nlargest(5,'gap')[["name", "abbr", "woba_round1", "woba_round3", "gap"]])

<bound method IndexOpsMixin.tolist of Index(['BB', 'BF', '2B', 'ER', 'ERA', 'GIDP', 'H', 'HR', 'IBB', 'IPOut', 'NP',
       'R', 'R_BS', 'R_H', 'R_L', 'R_SV', 'R_W', 'SB', 'SO', 'SP', '3B',
       'games', 'name', 'abbr', 'player_id', 'AVG', 'BABIP', 'BB9', 'BBK',
       'BBp', 'ERAplus', 'FIP', 'GBp', 'H9', 'HR9', 'Kp', 'LOBp', 'OBP',
       'OPSplus', 'SBp', 'SLG', 'SO9', 'WHIP', 'Whiffp', 'NegWPA', 'PosWPA',
       'RE24', 'WPA', 'IP', 'role', 'FIP_calc', 'team', 'woba_round1',
       'woba_round2', 'woba_round3', 'gap'],
      dtype='object')>
    name abbr  woba_round1  woba_round3       gap
135  伍立辰    象     0.303553     0.472184  0.168632
85   林詔恩    獅     0.256158     0.395922  0.139764
114  陳克羿    猿     0.292868     0.391495  0.098627
89   德保拉    象     0.259839     0.333401  0.073563
10   魔力藍    悍     0.296820     0.348214  0.051394


In [40]:
!git add .
!git commit -m "Day 9: 拆分數據（左右投打、第幾輪）"

[main fc6e1e7] Day 9: 拆分數據（左右投打、第幾輪）
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 8 files changed, 293 insertions(+), 399 deletions(-)
 delete mode 100644 .virtual_documents/D02.ipynb
 delete mode 100644 .virtual_documents/D03.ipynb
 delete mode 100644 .virtual_documents/D04.ipynb
 create mode 100644 src/splits.py


In [41]:
%%writefile -a src/splits.py

BREAKING = {"SL", "CU", "CH", "FO", "FS", "SC", "SV", "KC"}  # 變化球與變速球
SWING = {"SW", "F", "FT", "H"}                               # 有出棒的投球結果


Appending to src/splits.py


In [48]:
%%writefile -a src/splits.py

def batter_hand(pa):
    share=pa.groupby('batter_id')['b_hand'].value_counts(normalize=True).unstack(fill_value=0)
    def label(row):
        if row.get('L',0)>=0.1 and row.get('R',0)>=0.1:
            return 'S'
        return 'L' if row.get('L',0)>row.get('R',0) else 'R'
    return share.apply(label,axis=1).rename('bats').reset_index()

def pitcher_hand(pa):
    return pa.groupby('pitcher_id')['p_hand'].agg(lambda x:x.mode().iat[0]).rename('throws').reset_index()


Appending to src/splits.py


In [55]:
%%writefile -a src/splits.py

def breaking_whiff(pa):
    rows=[]
    for bid, types, codes in pa[['batter_id','pitch_types','pitch_codes']].itertuples(index=False):
        for t, c in zip(str(types).split(','),str(codes).split(',')):
            if t in BREAKING and c in SWING:
                rows.append((bid,c=='SW'))
    df=pd.DataFrame(rows,columns=['batter_id','whiff'])
    out=df.groupby('batter_id')['whiff'].agg(swings='size',whiff_rate='mean').reset_index()
    return out


Appending to src/splits.py


In [73]:
%%writefile -a src/splits.py

def pitch_mix(pa):
    rows=[]
    for pid,types in pa[['pitcher_id','pitch_types']].itertuples(index=False):
        for t in str(types).split(','):
            if t and t!='nan':
                rows.append((pid,t))
    df_pit=pd.DataFrame(rows,columns=['pitcher_id','pitch'])
    mix=df_pit.groupby('pitcher_id')['pitch'].value_counts(normalize=True).rename('share').reset_index()
    count=mix[mix['pitch'].isin(BREAKING)].sort_values('share',ascending=False)
    main=count.groupby('pitcher_id').head(1).rename(columns={'pitch':'main_breaking','share':'main_breaking_share'})
    return mix, main


Appending to src/splits.py


In [ ]:
%%writefile -a src/splits.py

def build_splits(pa, scale):
    pa = prepare(pa, scale)
    # 打者：對左投、右投
    bvs = summarize(pa, ["batter_id", "p_hand"]).pivot(index="batter_id", columns="p_hand",
                                                      values=["PA", "wOBA"])
    bvs.columns = [f"{a}_vs{b}" for a, b in bvs.columns]
    batters = (bvs.reset_index().merge(batter_hand(pa), on="batter_id", how="left")
               .merge(breaking_whiff(pa), on="batter_id", how="left"))
    # 投手：對左打、右打，以及第 1、2、3 輪打線
    pvs = summarize(pa, ["pitcher_id", "b_hand"]).pivot(index="pitcher_id", columns="b_hand",
                                                       values=["PA", "wOBA"])
    pvs.columns = [f"{a}_vs{b}" for a, b in pvs.columns]
    tto = summarize(pa, ["pitcher_id", "pa_round"]).pivot(index="pitcher_id", columns="pa_round",
                                                         values="wOBA")
    tto.columns = [f"wOBA_round{int(c)}" for c in tto.columns]
    _, main = pitch_mix(pa)
    pitchers = (pvs.reset_index().merge(tto.reset_index(), on="pitcher_id", how="left")
                .merge(pitcher_hand(pa), on="pitcher_id", how="left")
                .merge(main, on="pitcher_id", how="left"))
    league = {"wOBA": pa["woba_val"].mean()}
    return batters, pitchers, league


In [ ]:
%%writefile scripts/update_data.py
"""一鍵更新：抓資料 → 清理 → 拆分數據。用法：python scripts/update_data.py"""
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))
import pandas as pd

from src import clean, fetch, splits

clean.main()
const = pd.read_csv(ROOT / "data/clean/league_constants.csv", index_col=0).iloc[:, 0]

pitchers = pd.read_csv(ROOT / "data/clean/pitchers.csv")
frames = [fetch.fetch_pitcher_pa(pid) for pid in pitchers["player_id"]]
pa = pd.concat(frames, ignore_index=True)
pa.to_csv(ROOT / "data/clean/pa.csv", index=False)

b_split, p_split, lg = splits.build_splits(pa, const["woba_scale"])
b_split.to_csv(ROOT / "data/clean/batter_splits.csv", index=False)
p_split.to_csv(ROOT / "data/clean/pitcher_splits.csv", index=False)
print(f"逐打席 {len(pa)} 筆；聯盟 wOBA {lg['wOBA']:.3f}")


In [1]:
!python scripts/update_data.py

半季戰績來源：中職官網；交叉比對全部一致
球隊 6、打者 183、投手 181；wOBA 倍數 = 1.001、FIP 常數 = 3.02
逐打席 27268 筆；聯盟 wOBA 0.319
